# TACO YOLO26 Training (Google Colab)

Pre-training notebook used to fine-tune the YOLO26 models for the thesis
*"Benchmarking ML Inference Strategies on Edge Devices"* (TU Wien).

This is the notebook used to fine-tune the YOLO26 models on Google Colab (T4 GPU runtime), lightly cleaned for release.
It downloads the [TACO](https://github.com/pedropro/TACO) dataset, collapses all
annotations to a single `litter` class, converts to YOLO format with an 80/10/10
train/val/test split (`seed=42`), and fine-tunes a YOLO26 model for 50 epochs.

Notes for the reader:
- The three benchmarked weights (`yolo26n` / `yolo26s` / `yolo26m`) were each produced by
  changing the `used_model` variable in the training and testing cells and re-running.
- The inline comment "Cloud tier (AWS)" reflects an earlier plan; the final cloud tier
  is a **Hugging Face Space (NVIDIA T4)**, not AWS. Kept here as originally written.
- The three fine-tuned weights and their tier mapping are described in [`../models/README.md`](../models/README.md).
- The exported `taco_test.zip` is the fixed 150-image test split used across all benchmark
  configurations (`data/content/taco_yolo/`).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install "ultralytics>=8.4.0" -q

# Download Taco Dataset

This can take some time just be patient.

In [ ]:
import os

if not os.path.exists('/content/TACO/data/annotations.json'):
    !git clone https://github.com/pedropro/TACO.git
    %cd TACO
    !pip install -r requirements.txt -q
    !python download.py

# Map TACO lable to one "litter" label

In [ ]:
import json, os, shutil, random
from pathlib import Path
from collections import defaultdict

ANN_PATH = '/content/TACO/data/annotations.json'
IMG_DIR  = '/content/TACO/data'
OUT_DIR  = Path('/content/taco_yolo')

with open(ANN_PATH) as f:
    coco = json.load(f)

img_id_to_info = {i['id']: i for i in coco['images']}

anns_by_img = defaultdict(list)
for ann in coco['annotations']:
    anns_by_img[ann['image_id']].append(ann)

valid_ids = [
    img_id for img_id, info in img_id_to_info.items()
    if os.path.exists(os.path.join(IMG_DIR, info['file_name']))
]

random.seed(42)
random.shuffle(valid_ids)
n = len(valid_ids)
splits = {
    'train': valid_ids[:int(n*0.8)],
    'val':   valid_ids[int(n*0.8):int(n*0.9)],
    'test':  valid_ids[int(n*0.9):]
}

for split, ids in splits.items():
    (OUT_DIR / split / 'images').mkdir(parents=True, exist_ok=True)
    (OUT_DIR / split / 'labels').mkdir(parents=True, exist_ok=True)
    for img_id in ids:
        info      = img_id_to_info[img_id]
        src       = Path(IMG_DIR) / info['file_name']
        W, H      = info['width'], info['height']
        flat_name = info['file_name'].replace('/', '_')

        shutil.copy(src, OUT_DIR / split / 'images' / flat_name)

        lines = []
        for ann in anns_by_img[img_id]:
            if ann.get('iscrowd', 0):
                continue
            x, y, w, h = ann['bbox']
            cx = max(0.0, min(1.0, (x + w / 2) / W))
            cy = max(0.0, min(1.0, (y + h / 2) / H))
            nw = max(0.0, min(1.0, w / W))
            nh = max(0.0, min(1.0, h / H))
            lines.append(f'0 {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}')

        with open(OUT_DIR / split / 'labels' / (Path(flat_name).stem + '.txt'), 'w') as lf:
            lf.write('\n'.join(lines))

In [ ]:
yaml_content = """path: /content/taco_yolo
train: train/images
val:   val/images
test:  test/images

nc: 1
names:
  0: litter
"""

with open('/content/taco_yolo/data.yaml', 'w') as f:
    f.write(yaml_content)

## Zip and download test split

In [ ]:
# zip test split
!zip -r taco_test.zip /content/taco_yolo/test

In [ ]:
from google.colab import files
files.download('taco_test.zip')

# Train YOLO model on TACO

Choose different models according to https://github.com/ultralytics/ultralytics

In [ ]:
from ultralytics import YOLO

n = 'yolo26n'  # nano  → Device tier (Raspberry Pi)
s = 'yolo26s'  # small → Edge tier   (Lenovo Mini PC)
m = 'yolo26m'  # medium → Cloud tier  (AWS)
l = 'yolo26l'
x = 'yolo26x'

used_model = n

model = YOLO(f'{used_model}.pt')

results = model.train(
    data     = '/content/taco_yolo/data.yaml',
    epochs   = 50,
    imgsz    = 640,
    batch    = 16,
    lr0      = 0.001,
    device   = 0,
    seed     = 42,
    project  = '/content/drive/MyDrive/litter_detection',
    name     = used_model,
    exist_ok = True,
)

print(f"mAP50:    {results.results_dict['metrics/mAP50(B)']:.3f}")
print(f"mAP50-95: {results.results_dict['metrics/mAP50-95(B)']:.3f}")

# Testing model

In [ ]:
from ultralytics import YOLO

n = 'yolo26n'  # nano  → Device tier (Raspberry Pi)
s = 'yolo26s'  # small → Edge tier   (Lenovo Mini PC)
m = 'yolo26m'  # medium → Cloud tier  (AWS)
l = 'yolo26l'
x = 'yolo26x'

used_model = n

print(f"Model used for test {used_model}")

model = YOLO(f'/content/drive/MyDrive/litter_detection/{used_model}/weights/best.pt')

metrics = model.val(
    data='/content/taco_yolo/data.yaml',
    split='test',
    verbose=True
)

print(f"mAP50:     {metrics.box.map50:.3f}")
print(f"mAP50-95:  {metrics.box.map:.3f}")
print(f"Precision: {metrics.box.mp:.3f}")
print(f"Recall:    {metrics.box.mr:.3f}")